# Simulation benchmarking plots (Fig. 2B–D)

This notebook reproduces the original Python plotting and summary logic for Fig. 2B–D from simulation results generated and postprocessed separately in R.

All plotting inputs are loaded from Additional File 3. The standalone historical workbooks were verified cell-for-cell against its corresponding sheets and are not required publication inputs.


## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

from scale_aware_st import RepositoryConfig

config = RepositoryConfig.from_env()
additional_file_3 = config.additional_data_dir / 'Additional_File_3.xlsx'
figure_output_dir = config.results_dir / 'figure2_simulations'
figure_output_dir.mkdir(parents=True, exist_ok=True)

fig2b_output = figure_output_dir / 'signal_hybrid.png'


def add_method_labels(df, scanpy_label='Wilcoxon'):
    """Add manuscript-facing labels while preserving figure-specific Scanpy wording."""
    out = df.copy()
    out['label'] = np.select(
        [
            out['method'].eq('ALDEx') & out['condition'].eq('observed_tss'),
            out['method'].eq('ALDEx') & out['condition'].eq('observed_it'),
            out['method'].eq('Scanpy'),
            out['method'].eq('MAST'),
            out['method'].eq('NB'),
        ],
        [
            'ALDEx-TSS',
            'ALDEx-IT',
            scanpy_label,
            'MAST',
            'Negative Binomial',
        ],
        default=np.nan,
    )
    return out.loc[out['label'].notna()].copy()


## 1. Fig. 2B — Signal benchmark

Summarizes empirical FDR, power, positive predictive value, and F0.5 across simulated sample sizes. The source method `Scanpy` is displayed as `Wilcoxon`.

In [ ]:
signal_df = pd.read_excel(additional_file_3, sheet_name='Performance_summary')
signal_df = add_method_labels(signal_df, scanpy_label='Wilcoxon')

label_order = ['ALDEx-TSS', 'ALDEx-IT', 'MAST', 'Negative Binomial', 'Wilcoxon']

color_map = {
    'ALDEx-TSS': '#1f77b4',
    'ALDEx-IT': '#17becf',
    'MAST': '#ff7f0e',
    'Negative Binomial': '#2ca02c',
    'Wilcoxon': '#d62728',
}

zorder_map = {
    'ALDEx-TSS': 5,
    'ALDEx-IT': 4,
    'MAST': 3,
    'Negative Binomial': 2,
    'Wilcoxon': 1,
}


def mean_ci(df, value_col, group_cols=('label', 'N_batches')):
    out = (
        df.groupby(list(group_cols))[value_col]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    out['se'] = out['std'] / np.sqrt(out['count'])
    out['ci95'] = 1.96 * out['se']
    return out


def plot_metric(
    ax,
    summary_df,
    x_col,
    y_col='mean',
    ci_col='ci95',
    hue_col='label',
    title='',
    ylabel='',
    hline=None,
    ylim=None,
    floor_zero=False,
):
    offset_map = {
        'ALDEx-TSS': -0.5,
        'ALDEx-IT': -0.25,
        'MAST': 0.0,
        'Negative Binomial': 0.25,
        'Wilcoxon': 0.5,
    }

    for label in label_order:
        sub = summary_df.loc[summary_df[hue_col].eq(label)].sort_values(x_col)
        if sub.empty:
            continue

        x = sub[x_col].to_numpy(dtype=float) + offset_map[label]
        y = sub[y_col].to_numpy(dtype=float)
        ci = sub[ci_col].fillna(0).to_numpy(dtype=float)
        y_plot = np.where(np.isclose(y, 0), 1e-4, y) if floor_zero else y

        ax.plot(
            x,
            y_plot,
            color=color_map[label],
            marker='o',
            linestyle='-',
            linewidth=1.5,
            markersize=5,
            label=label,
            zorder=zorder_map[label],
        )
        ax.fill_between(
            x,
            y_plot - ci,
            y_plot + ci,
            color=color_map[label],
            alpha=0.12,
            zorder=zorder_map[label] - 0.1,
        )

    if hline is not None:
        ax.axhline(hline, linestyle='--', linewidth=1, color='black')

    ax.set_title(title)
    ax.set_xlabel('N batches')
    ax.set_ylabel(ylabel)

    if ylim is not None:
        ax.set_ylim(*ylim)


signal_fdr = mean_ci(signal_df, 'emp_fdr')
signal_power = mean_ci(signal_df, 'power')
signal_ppv = mean_ci(signal_df, 'ppv')
signal_f05 = mean_ci(signal_df, 'f0_5')

fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))

plot_metric(
    axes[0, 0], signal_fdr, x_col='N_batches',
    title='Signal: empirical FDR', ylabel='Empirical FDR',
    hline=0.05, ylim=(-0.02, 1.1), floor_zero=True,
)
plot_metric(
    axes[0, 1], signal_power, x_col='N_batches',
    title='Signal: power', ylabel='Power', ylim=(-0.02, 1.1),
)
plot_metric(
    axes[1, 0], signal_ppv, x_col='N_batches',
    title='Signal: PPV', ylabel='PPV',
    ylim=(-0.02, 1.1), floor_zero=True,
)
plot_metric(
    axes[1, 1], signal_f05, x_col='N_batches',
    title='Signal: F0.5', ylabel='F0.5',
    ylim=(-0.02, 1.1), floor_zero=True,
)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)

for ax in axes.ravel():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

fig.suptitle('Signal benchmark', y=0.98)
fig.tight_layout(rect=[0, 0.06, 1, 0.96])

# fig.savefig(fig2b_output, dpi=300, bbox_inches='tight')
plt.show()


## 2. Load per-gene simulation results for Fig. 2C

The source method `Scanpy` is displayed as `Wilcoxon`, matching the original Fig. 2C code.

In [ ]:
# Additional File 3 contains the verified combined gene-level table.
per_gene_plot_df = pd.read_excel(additional_file_3, sheet_name='Gene_level_results')
per_gene_plot_df = add_method_labels(per_gene_plot_df, scanpy_label='Wilcoxon')


## 3. Fig. 2C — Null p-value distributions

In [ ]:
N_target = 32

df_plot = per_gene_plot_df.loc[
    per_gene_plot_df['injected'].eq(False)
    & per_gene_plot_df['N_batches'].eq(N_target)
].copy()

# Preserve the original selection logic: first restrict after sorting, then retain
# the first simulation represented within each method.
df_plot = df_plot.sort_values('sim_id')
df_plot = df_plot.groupby('label').head(len(df_plot['gene'].unique()))
first_sim_id = df_plot.groupby('label')['sim_id'].transform('first')
df_plot = df_plot.loc[df_plot['sim_id'].eq(first_sim_id)].reset_index(drop=True)

methods = [label for label in label_order if label in df_plot['label'].unique()]
fig, axes = plt.subplots(1, len(methods), figsize=(2.5 * len(methods), 3), sharey=False)

if len(methods) == 1:
    axes = [axes]

for ax, label in zip(axes, methods):
    pvals = df_plot.loc[df_plot['label'].eq(label), 'pval'].dropna().to_numpy()
    is_aldex = 'ALDEx' in label
    binrange = (0, 0.5) if is_aldex else (0, 0.055)

    sns.histplot(
        pvals,
        bins=50,
        binrange=binrange,
        color=color_map[label],
        ax=ax,
    )

    ax.axvline(0.05, linestyle='--', color='black', linewidth=1)
    ax.set_title(label)
    ax.set_xlabel('p-value')
    ax.set_ylabel('Count')

    if not is_aldex:
        ax.tick_params(axis='both', colors='red')
        ax.set_ylim(0, 120)
    else:
        ax.set_ylim(0, 25)

fig.suptitle(f'Null p-value distributions (N={N_target})')
fig.tight_layout()
plt.show()


## 4. Effect-size correlation summary

Calculates per-simulation correlations and RMSE for injected genes. The original, unmodified `per_gene` sheet is written back to the output workbook alongside the summary.

In [ ]:
# Load the verified combined gene-level table without manuscript-facing labels.
per_gene_raw_df = pd.read_excel(additional_file_3, sheet_name='Gene_level_results')
injected_df = per_gene_raw_df.loc[per_gene_raw_df['injected'].eq(True)].copy()


def safe_corr(x, y, kind='pearson'):
    valid = x.notna() & y.notna()
    x = x.loc[valid]
    y = y.loc[valid]

    if len(x) < 3 or x.nunique() < 2 or y.nunique() < 2:
        return np.nan

    if kind == 'pearson':
        return pearsonr(x, y)[0]
    if kind == 'spearman':
        return spearmanr(x, y)[0]

    raise ValueError("kind must be 'pearson' or 'spearman'")


correlation_summary = (
    injected_df
    .groupby(['method', 'condition', 'N_batches', 'sim_id'])
    .apply(
        lambda g: pd.Series({
            'n_injected_genes': g.shape[0],
            'pearson_true_vs_estimated': safe_corr(g['true_effect'], g['coef'], 'pearson'),
            'spearman_true_vs_estimated': safe_corr(g['true_effect'], g['coef'], 'spearman'),
            'mean_true_effect': g['true_effect'].mean(),
            'mean_estimated_effect': g['coef'].mean(),
            'rmse': np.sqrt(np.mean((g['coef'] - g['true_effect']) ** 2)),
        })
    , include_groups=False)
    .reset_index()
)

published_correlation_summary = pd.read_excel(
    additional_file_3,
    sheet_name='Effect_size_correlations',
)

# Verify that the transparent recomputation above reproduces the deposited table.
comparison_columns = list(published_correlation_summary.columns)
pd.testing.assert_frame_equal(
    correlation_summary[comparison_columns].reset_index(drop=True),
    published_correlation_summary[comparison_columns].reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)


## 5. Fig. 2D — Effect-size recovery

The source method `Scanpy` is displayed as `Scanpy Wilcoxon` here, matching the original Fig. 2D code.

In [ ]:
# Reload the published gene-level table and use the Fig. 2D-specific label.
fig2d_df = pd.read_excel(additional_file_3, sheet_name='Gene_level_results')
fig2d_df = add_method_labels(fig2d_df, scanpy_label='Scanpy Wilcoxon')

fig2d_color_map = {
    'ALDEx-TSS': '#1f77b4',
    'ALDEx-IT': '#17becf',
    'MAST': '#ff7f0e',
    'Negative Binomial': '#2ca02c',
    'Scanpy Wilcoxon': '#d62728',
}
fig2d_label_order = list(fig2d_color_map.keys())

panel_specs = [
    {'N': 8, 'sim_id': 7},
    {'N': 32, 'sim_id': 23},
    {'N': 100, 'sim_id': 27},
]


def pick_genes_across_range(sub, n_genes=7):
    sub_unique = sub.drop_duplicates('gene').sort_values('true_effect')

    targets = np.linspace(
        sub_unique['true_effect'].min(),
        sub_unique['true_effect'].max(),
        n_genes,
    )

    selected = []
    for target in targets:
        idx = (sub_unique['true_effect'] - target).abs().idxmin()
        selected.append(idx)

    selected = sub_unique.loc[selected].drop_duplicates('gene')
    return selected['gene'].to_numpy()


lim = np.nanmax([
    np.abs(fig2d_df['true_effect']).max(),
    np.abs(fig2d_df['coef']).max(),
]) + 0.5

fig, axes = plt.subplots(
    len(panel_specs),
    1,
    figsize=(6, 5 * len(panel_specs)),
    sharex=True,
    sharey=False,
)

if len(panel_specs) == 1:
    axes = [axes]

for ax, spec in zip(axes, panel_specs):
    N = spec['N']
    sim_id = spec['sim_id']

    sub = fig2d_df.loc[
        fig2d_df['N_batches'].eq(N)
        & fig2d_df['sim_id'].eq(sim_id)
        & fig2d_df['injected'].eq(True)
    ].copy()

    if sub.empty:
        ax.set_title(f'N={N}, sim={sim_id} (no data)')
        continue

    selected_genes = pick_genes_across_range(sub, n_genes=7)
    sub = sub.loc[sub['gene'].isin(selected_genes)]

    marker_list = ['o', 's', '^', 'D', 'P', 'X', '*', 'v', '<', '>']
    gene_marker_map = {
        gene: marker_list[i]
        for i, gene in enumerate(selected_genes)
    }

    for label in fig2d_label_order:
        method_df = sub.loc[sub['label'].eq(label)]

        for _, row in method_df.iterrows():
            ax.scatter(
                row['true_effect'],
                row['coef'],
                color=fig2d_color_map[row['label']],
                marker=gene_marker_map[row['gene']],
                s=140,
                alpha=0.9,
                edgecolor='black',
                linewidth=0.6,
            )

    handles = [
        plt.Line2D(
            [0], [0],
            marker=gene_marker_map[gene],
            color='gray',
            linestyle='',
            markersize=10,
            label=gene,
        )
        for gene in selected_genes
    ]

    ax.legend(
        handles=handles,
        title='Genes',
        fontsize=11,
        title_fontsize=13,
        frameon=False,
        loc='upper left',
    )

    ax.plot(
        [-lim, lim],
        [-lim, lim],
        linestyle='--',
        color='black',
        linewidth=1.5,
    )

    ax.set_title(f'N = {N}, sim = {sim_id}', fontsize=15)
    ax.set_ylabel('Estimated effect size', fontsize=13)
    ax.tick_params(axis='both', labelsize=11)
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 5.5)

axes[-1].set_xlabel('True effect size', fontsize=13)
plt.tight_layout()
plt.show()
